In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Machine Learning model
from sklearn.ensemble import RandomForestClassifier

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"/content/predictive_maintenance_v3.csv")


df.head()

,timestamp,machine_id,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,rul_hours,failure_within_24h,failure_type,estimated_repair_cost
0,1/2/2024 0:07,1,CNC,0.73,49.87,4.56,22.2,906.9,idle,297.92,9.9,36.88,0,none,0
1,1/2/2024 0:28,1,CNC,0.98,49.86,5.13,22.3,855.9,idle,298.27,16.6,36.53,0,none,0
2,1/2/2024 0:42,1,CNC,0.93,45.58,4.64,25.4,857.8,idle,298.50,16.9,36.30,0,none,0
3,1/2/2024 0:51,1,CNC,0.85,40.11,4.70,20.3,858.3,idle,298.65,8.4,36.15,0,none,0
4,1/2/2024 0:55,1,CNC,0.74,46.49,5.12,23.5,917.4,idle,298.72,13.8,36.08,0,none,0


In [ ]:
print(
    df['failure_type'].value_counts()
)

failure_type
none              20093
bearing            1025
motor_overheat      973
electrical          655
hydraulic           571
Name: count, dtype: int64


In [ ]:
df.dtypes

,0
timestamp,object
machine_id,int64
machine_type,object
vibration_rms,float64
temperature_motor,float64
current_phase_avg,float64
pressure_level,float64
rpm,float64
operating_mode,object
hours_since_maintenance,float64


In [ ]:
columns = [
    'temperature_motor',
    'vibration_rms',
    'current_phase_avg',
    'pressure_level',
    'rpm'

]

for col in columns:

    median_0 = df.loc[
        df['failure_within_24h'] == 0,
        col
    ].median()

    median_1 = df.loc[
        df['failure_within_24h'] == 1,
        col
    ].median()

    df.loc[
        (df[col].isna()) &
        (df['failure_within_24h'] == 0),
        col
    ] = median_0

    df.loc[
        (df[col].isna()) &
        (df['failure_within_24h'] == 1),
        col
    ] = median_1

In [ ]:
# Filter Records with Actual Failures
df_failure = df[
    df['failure_type'] != 'none'
].copy()

In [ ]:
df_failure.drop(
    columns=[
        'timestamp',
        'machine_id',
        'failure_within_24h',
        'rul_hours',
        'estimated_repair_cost'
    ],
    inplace=True
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

enc_machine = LabelEncoder()
enc_machine.fit(df_failure['failure_type'])

print(dict(zip(
    enc_machine.classes_,
    enc_machine.transform(enc_machine.classes_)
)))

{'bearing': np.int64(0), 'electrical': np.int64(1), 'hydraulic': np.int64(2), 'motor_overheat': np.int64(3)}


In [ ]:
# Encode Categorical Variables
from sklearn.preprocessing import LabelEncoder

machine_encoder = LabelEncoder()

df_failure['machine_type'] = (
    machine_encoder.fit_transform(
        df_failure['machine_type']
    )
)

operating_encoder = LabelEncoder()

df_failure['operating_mode'] = (
    operating_encoder.fit_transform(
        df_failure['operating_mode']
    )
)

failure_encoder = LabelEncoder()

df_failure['failure_type'] = (
    failure_encoder.fit_transform(
        df_failure['failure_type']
    )
)

In [ ]:
X = df_failure.drop(
    'failure_type',
    axis=1
)

y = df_failure['failure_type']

In [ ]:
# Split Dataset into Training and Testing Sets
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

In [ ]:
# Train Random Forest Classification Model
model_failure = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)

model_failure.fit(
    X_train,
    y_train
)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=5, min_samples_split=10,
                       n_estimators=200, random_state=42)

In [ ]:
y_pred = model_failure.predict(
    X_test
)

In [ ]:
# Evaluate Model Performance

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

print()

print(
    classification_report(
        y_test,
        y_pred
    )
)

Accuracy: 0.9658914728682171

              precision    recall  f1-score   support

           0       0.96      0.98      0.97       205
           1       0.96      0.98      0.97       131
           2       0.97      0.91      0.94       114
           3       0.97      0.98      0.98       195

    accuracy                           0.97       645
   macro avg       0.97      0.96      0.96       645
weighted avg       0.97      0.97      0.97       645



In [ ]:
# Save Trained Model

import joblib

joblib.dump(
    model_failure,
    'failure_type_model.pkl'
)

['failure_type_model.pkl']

In [ ]:
!pip install fastapi uvicorn pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("****************")

In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8001)
print(public_url)

NgrokTunnel: "https://30bf-34-29-120-97.ngrok-free.app" -> "http://localhost:8001"


In [ ]:

from fastapi import FastAPI
import joblib
import pandas as pd

app1 = FastAPI()

model = joblib.load("/content/failure_type_model.pkl")

@app1.post("/predict")
def predict(data: dict):

    df = pd.DataFrame([data])

    prediction = int(model.predict(df)[0])

    return {
        "failure_type": prediction
    }
print(app1.routes)


[Route(path='/openapi.json', name='openapi', methods=['GET', 'HEAD']), Route(path='/docs', name='swagger_ui_html', methods=['GET', 'HEAD']), Route(path='/docs/oauth2-redirect', name='swagger_ui_redirect', methods=['GET', 'HEAD']), Route(path='/redoc', name='redoc_html', methods=['GET', 'HEAD']), APIRoute(path='/predict', name='predict', methods=['POST'])]


In [ ]:

import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run():
    uvicorn.run(app1, host="0.0.0.0", port=8001)

Thread(target=run).start()


In [ ]:
import requests

payload = {
    "machine_type": 0,
    "vibration_rms": 1.24,
    "temperature_motor": 47.33,
    "current_phase_avg": 4.93,
    "pressure_level": 21.2,
    "rpm": 85.2,
    "operating_mode": 0,
    "hours_since_maintenance": 331.75,
    "ambient_temp": 8.7
}

response1 = requests.post(
    "https://d10e-136-116-60-59.ngrok-free.app/predict",
    json=payload
)

print(response1.status_code)
print(response1.text)


INFO:     Started server process [490]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


404
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="prelo